# P6-1 · Bài 8a · SFT và PEFT

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
# một dòng trong train.jsonl, dạng prompt/completion hội thoại
{"prompt": [{"role": "system", "content": "Trích xuất ý định và mã đơn. Chỉ trả JSON."},
            {"role": "user", "content": "Tôi muốn đổi size áo đơn DH1024"}],
 "completion": [{"role": "assistant", "content": "{\"intent\": \"đổi hàng\", \"order\": \"DH1024\"}"}]}

{'prompt': [{'role': 'system', 'content': 'Trích xuất ý định và mã đơn. Chỉ trả JSON.'}, {'role': 'user', 'content': 'Tôi muốn đổi size áo đơn DH1024'}], 'completion': [{'role': 'assistant', 'content': '{"intent": "đổi hàng", "order": "DH1024"}'}]}


## Bài tập 5: Định dạng dữ liệu hội thoại {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json, pandas as pd

# file mẫu 2 dòng để chạy thử (dữ liệu thật: vài trăm tới vài nghìn dòng)
pd.DataFrame({"text": ["Đơn DH1024 giao sai màu ghế, mình cần đổi gấp trước thứ 7.", "Cho hỏi bàn trang điểm còn hàng không?"],
              "json_label": ['{"intent": "đổi hàng", "order": "DH1024", "product": "ghế", "urgent": true}',
                             '{"intent": "hỏi hàng", "order": null, "product": "bàn trang điểm", "urgent": false}']}
             ).to_csv("emails.csv", index=False)

SYSTEM = "Trích xuất ý định, mã đơn, sản phẩm và mức khẩn từ email khách hàng. Chỉ trả JSON."
df = pd.read_csv("emails.csv")
with open("train.jsonl", "w", encoding="utf-8") as f:
    for _, r in df.iterrows():
        f.write(json.dumps({
            "prompt": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": r["text"]}],
            "completion": [{"role": "assistant", "content": r["json_label"]}],
        }, ensure_ascii=False) + "\n")
print(open("train.jsonl", encoding="utf-8").readline())

{"prompt": [{"role": "system", "content": "Trích xuất ý định, mã đơn, sản phẩm và mức khẩn từ email khách hàng. Chỉ trả JSON."}, {"role": "user", "content": "Đơn DH1024 giao sai màu ghế, mình cần đổi gấp trước thứ 7."}], "completion": [{"role": "assistant", "content": "{\"intent\": \"đổi hàng\", \"order\": \"DH1024\", \"product\": \"ghế\", \"urgent\": true}"}]}



In [ ]:
# Cần mạng: tải tokenizer Qwen từ Hugging Face (chạy trên Colab)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
sample = json.loads(open("train.jsonl", encoding="utf-8").readline())
print(tok.apply_chat_template(sample["prompt"] + sample["completion"], tokenize=False))